# Environment Setup for Azure AI Foundry Workshop

This notebook will guide you through setting up your environment for the Azure AI Foundry workshop.

## Prerequisites
- Python 3.8 or later
- Azure subscription with AI services access
- Basic Python knowledge

## Azure Authentication Setup
First, we'll verify our Azure credentials and setup.

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
import os

# Initialize Azure credentials
try:
    credential = DefaultAzureCredential()
    print("✓ Successfully initialized DefaultAzureCredential")
except Exception as e:
    print(f"× Error initializing credentials: {str(e)}")

## Initialize AI Project Client

> **Note:** Before proceeding, ensure you:
> 1. Copy your `.env.example` file to `.env`
> 2. Set `PROJECT_ENDPOINT` in your `.env` file
> 3. Have a Project already provisioned in Microsoft Foundry

You can find your **Project endpoint** on the [Microsoft Foundry](https://ai.azure.com) Home page (it looks like `https://<resource>.services.ai.azure.com/api/projects/<project>`).


## Understanding AIProjectClient

The AIProjectClient is a key component for interacting with Azure AI services that:

- **Manages Connections**: Lists and accesses Azure AI resources like OpenAI models
- **Handles Authentication**: Securely connects using Azure credentials  
- **Enables Model Access**: Provides interfaces to use AI models and deployments
- **Manages Project Settings**: Controls configurations for your Azure AI project

The client requires:
- A **project endpoint** (`PROJECT_ENDPOINT`, from the Foundry Home page)
- Valid Azure credentials (`DefaultAzureCredential`)


In [ ]:
from dotenv import load_dotenv
from pathlib import Path

# Load environment variables
notebook_path = Path().absolute()
parent_dir = notebook_path.parent
load_dotenv(parent_dir / '.env')

# Initialize the endpoint-based AIProjectClient (New Foundry)
try:
    client = AIProjectClient(
        endpoint=os.getenv("PROJECT_ENDPOINT"),
        credential=credential,
    )
    print("✓ Successfully initialized AIProjectClient")
except Exception as e:
    print(f"× Error initializing client: {str(e)}")

## Verify Access to Models and Connections
Let's verify we can access the connections specified in the [prerequisites](../README.md#-prerequisites).

The cell below uses the endpoint-based `AIProjectClient` to enumerate the connections defined in your Foundry project. Look for:
- An **Azure OpenAI** connection, which provides access to your deployed models (for example, the `gpt-5.4` chat deployment and an embedding model)
- An **Azure AI Search** connection for vector search and knowledge retrieval

Listing the connections confirms that the components needed to build our AI applications are provisioned and reachable from your project.

In [ ]:
# List the properties of all connections in the project
connections = list(client.connections.list())
print(f"====> Listing of all connections (found {len(connections)}):")
for connection in connections:
    print(f"- {connection.name} | type={connection.type} | default={connection.is_default}")

# List only the Azure OpenAI connections (connection_type accepts the string category)
openai_connections = list(client.connections.list(connection_type="AzureOpenAI"))
print(f"\n====> Azure OpenAI connections (found {len(openai_connections)}):")
for connection in openai_connections:
    print(f"- {connection.name} | {connection.id}")

## Validate Model and Search Connections
The cell below validates that we have properly provisioned and connected to:
1. Azure OpenAI models through our Azure OpenAI connection
2. Azure AI Search through our Azure AI Search connection

Both of these services will be essential for building our AI applications. The OpenAI models will provide the core language capabilities, while Azure AI Search will enable efficient information retrieval and knowledge base functionality.



In [ ]:
# List all connections and check for the ones we need (Azure AI Search + Azure OpenAI)
search_conn_id = ""
openai_conn_id = ""

for conn in client.connections.list():
    conn_type = str(conn.type)  # e.g. "ConnectionType.COGNITIVE_SEARCH" or "CognitiveSearch"
    if "Search" in conn_type:
        search_conn_id = conn.id
    elif "OpenAI" in conn_type:
        openai_conn_id = conn.id

print("\n====> Connection IDs found:")
if not search_conn_id:
    print("Azure AI Search: Not found - Please create an Azure AI Search connection")
else:
    print(f"Azure AI Search: {search_conn_id}")

if not openai_conn_id:
    print("Azure OpenAI: Not found - Please create an Azure OpenAI connection")
else:
    print(f"Azure OpenAI: {openai_conn_id}")